In [ ]:
import os
import numpy as np
from copy import deepcopy
from pymatgen.core import Lattice
from pymatgen.io.pwscf import PWInput

## Генерация входных файлов

In [ ]:
# директория для входных файлов
INPUT_DIR = "inputs"
os.makedirs(INPUT_DIR, exist_ok=True)

input_data = PWInput.from_file('PWscf/pwscf.in')
structure = input_data.structure.copy()

idx = 0
input_data.write_file(os.path.join(INPUT_DIR, f'{idx:05}.in'))
idx += 1


## Блок 1. Деформации

In [15]:
count = 11
strain_step = 0.01

for i in range(count):
    s = i - count // 2
    if s == 0:
        continue

    # изотропная
    input_data.structure.lattice = Lattice(
        structure.lattice.matrix @ np.diag([1 + strain_step*s]*3)
    )
    input_data.write_file(os.path.join(INPUT_DIR, f'{idx:05}.in'))
    idx += 1

    # одноосная
    input_data.structure.lattice = Lattice(
        structure.lattice.matrix @ [[1 + strain_step*s, 0, 0],
                                    [0, 1, 0],
                                    [0, 0, 1]]
    )
    input_data.write_file(os.path.join(INPUT_DIR, f'{idx:05}.in'))
    idx += 1

    # сдвиг
    input_data.structure.lattice = Lattice(
        structure.lattice.matrix @ [[1, strain_step*s, 0],
                                    [0, 1, 0],
                                    [0, 0, 1]]
    )
    input_data.write_file(os.path.join(INPUT_DIR, f'{idx:05}.in'))
    idx += 1


## Случайные деформации

In [16]:
for _ in range(30):
    deformation = np.eye(3) + np.random.uniform(-0.05, 0.05, (3, 3))
    input_data.structure.lattice = Lattice(
        structure.lattice.matrix @ deformation
    )
    input_data.write_file(os.path.join(INPUT_DIR, f'{idx:05}.in'))
    idx += 1


## Блок 2. Суперячейка и смещения атомов

In [17]:
reps = 3
structure.make_supercell([reps]*3)

input_data.structure = structure.copy()
input_data.kpoints_grid = [
    (input_data.kpoints_grid[i] + reps - 1) // reps for i in range(3)
]
input_data.sections['system']['nat'] = len(structure)


In [18]:
for _ in range(25):
    input_data.structure = structure.copy()
    input_data.structure.perturb(0.1)
    input_data.write_file(os.path.join(INPUT_DIR, f'{idx:05}.in'))
    idx += 1


## Блок 3. Дефекты (вакансии)

In [19]:
for _ in range(15):
    defective = deepcopy(structure)
    defective.perturb(0.1)

    n_remove = np.random.randint(1, 3)
    remove_ids = np.random.choice(len(defective), n_remove, replace=False)

    for i in sorted(remove_ids, reverse=True):
        defective.remove_sites([i])

    input_data.structure = defective
    input_data.sections['system']['nat'] = len(defective)
    input_data.write_file(os.path.join(INPUT_DIR, f'{idx:05}.in'))
    idx += 1


## Запуск всех расчётов Quantum ESPRESSO

In [1]:
%%writefile run_qe.sh
#!/bin/bash

INPUT_DIR=inputs
OUTPUT_DIR=outputs
PSEUDO_DIR=PWscf

mkdir -p $OUTPUT_DIR

for infile in $INPUT_DIR/*.in; do
    name=$(basename $infile .in)
    mpirun -np 16 pw.x < $infile > $OUTPUT_DIR/$name.out
done


Overwriting run_qe.sh


In [25]:
!chmod +x run_qe.sh
!./run_qe.sh


^C


In [28]:
import glob
from pymatgen.io.pwscf import PWInput
from monty.serialization import dumpfn

ry_to_ev = 13.6056980659
a_to_bohr = 0.529177210903

data = []

for infile in sorted(glob.glob('inputs/*.in')):
    outfile = infile.replace('inputs', 'outputs').replace('.in', '.out')

    input_base = PWInput.from_file(infile)
    forces = []

    with open(outfile) as f:
        for line in f:
            if line.startswith('!'):
                energy = float(line.split()[-2]) * ry_to_ev
            if line.lstrip().startswith('atom') and 'force =' in line:
                parts = line.split('force =')[-1].split()
                fxyz = [float(parts[i]) * ry_to_ev * a_to_bohr for i in range(3)]
                forces.append(fxyz)

    data.append({
        'structure': input_base.structure,
        'energy': energy,
        'forces': forces
    })

dumpfn(data, 'my.json')
print(f'Saved {len(data)} structures to my.json')


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/00011.out'